# Notebook 07 Part 2: PySpark Joins

## Lecture 13: Inner, Left, Right, Full, Semi and Anti Joins

```

## 1. What Is a Join?

A join combines rows from two DataFrames using one or more common columns.

Example:

```text
Customers DataFrame + Orders DataFrame
                 ↓ join on customer_id
Customer details with order details
```

## 2. Join Types

| Join type | Simple meaning |
|---|---|
| `inner` | Keep matching rows from both DataFrames |
| `left` | Keep all left rows and matching right rows |
| `right` | Keep all right rows and matching left rows |
| `full` | Keep all rows from both DataFrames |
| `left_semi` | Keep left rows having a match |
| `left_anti` | Keep left rows having no match |

## 3. Windows Local Setup

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## 4. Create SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark_Joins") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 5. Create Customers DataFrame

In [3]:
customers_data = [
    (1, "Amit", "Delhi"),
    (2, "Priya", "Mumbai"),
    (3, "Rahul", "Pune"),
    (4, "Sneha", "Delhi"),
    (5, "Karan", "Jaipur")
]

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name", "city"]
)

customers_df.show()

+-----------+-------------+------+
|customer_id|customer_name|  city|
+-----------+-------------+------+
|          1|         Amit| Delhi|
|          2|        Priya|Mumbai|
|          3|        Rahul|  Pune|
|          4|        Sneha| Delhi|
|          5|        Karan|Jaipur|
+-----------+-------------+------+



## 6. Create Orders DataFrame

Notice:

```text
Customer 1 has two orders.
Customer 5 has no order.
Customer 6 exists only in the orders table.
```

In [4]:
orders_data = [
    (101, 1, 5000),
    (102, 2, 7000),
    (103, 1, 3000),
    (104, 4, 9000),
    (105, 6, 4500)
]

orders_df = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "amount"]
)

orders_df.show()

+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|     101|          1|  5000|
|     102|          2|  7000|
|     103|          1|  3000|
|     104|          4|  9000|
|     105|          6|  4500|
+--------+-----------+------+



## 7. Basic Join Syntax

```python
left_df.join(
    right_df,
    on="common_column",
    how="join_type"
)
```

# 8. Inner Join

Keeps only records that match in both DataFrames.

In [ ]:
inner_join_df = customers_df.join(
    orders_df,
    on="customer_id",
    how="inner"
)

inner_join_df.show()

Customer 1 appears twice because customer 1 has two matching orders.

# 9. Left Join

Keeps all rows from the left DataFrame and matching rows from the right DataFrame.

In [ ]:
left_join_df = customers_df.join(
    orders_df,
    on="customer_id",
    how="left"
)

left_join_df.show()

Customers without orders remain, but order columns contain `null`.

# 10. Right Join

Keeps all rows from the right DataFrame and matching rows from the left DataFrame.

In [ ]:
right_join_df = customers_df.join(
    orders_df,
    on="customer_id",
    how="right"
)

right_join_df.show()

Order for customer 6 remains even though customer details are unavailable.

# 11. Full Outer Join

Keeps all rows from both DataFrames.

In [ ]:
full_join_df = customers_df.join(
    orders_df,
    on="customer_id",
    how="full"
)

full_join_df.show()

## 12. Comparison

| Join | Matching rows | Unmatched left rows | Unmatched right rows |
|---|---|---|---|
| Inner | Yes | No | No |
| Left | Yes | Yes | No |
| Right | Yes | No | Yes |
| Full | Yes | Yes | Yes |

# 13. Left Semi Join

Keeps rows from the left DataFrame that have at least one match in the right DataFrame.

Only columns from the left DataFrame are returned.

In [ ]:
customers_with_orders = customers_df.join(
    orders_df,
    on="customer_id",
    how="left_semi"
)

customers_with_orders.show()

# 14. Left Anti Join

Keeps rows from the left DataFrame that do not have a match in the right DataFrame.

In [ ]:
customers_without_orders = customers_df.join(
    orders_df,
    on="customer_id",
    how="left_anti"
)

customers_without_orders.show()

## 15. Semi vs Anti

| Join | Result |
|---|---|
| Left semi | Left rows with a match |
| Left anti | Left rows without a match |

# 16. Join Using an Explicit Condition

In [ ]:
from pyspark.sql.functions import col

condition_join_df = customers_df.join(
    orders_df,
    customers_df.customer_id == orders_df.customer_id,
    "inner"
)

condition_join_df.show()

When using an explicit condition, both join columns may remain in the result.

# 17. Handle Duplicate Column Names with Aliases

In [ ]:
clean_join_df = customers_df.alias("c").join(
    orders_df.alias("o"),
    col("c.customer_id") == col("o.customer_id"),
    "inner"
).select(
    col("c.customer_id"),
    col("c.customer_name"),
    col("c.city"),
    col("o.order_id"),
    col("o.amount")
)

clean_join_df.show()

Use aliases when both DataFrames contain columns with the same names.

# 18. Join When Column Names Are Different

In [ ]:
payments_data = [
    (1001, 1, "UPI"),
    (1002, 2, "Card"),
    (1003, 4, "Cash"),
    (1004, 7, "UPI")
]

payments_df = spark.createDataFrame(
    payments_data,
    ["payment_id", "cust_id", "payment_mode"]
)

payments_df.show()

In [ ]:
customer_payments_df = customers_df.alias("c").join(
    payments_df.alias("p"),
    col("c.customer_id") == col("p.cust_id"),
    "inner"
).select(
    col("c.customer_id"),
    col("c.customer_name"),
    col("p.payment_id"),
    col("p.payment_mode")
)

customer_payments_df.show()

# 19. Join on Multiple Columns

Sometimes one column is not enough to identify a correct match.

In [ ]:
location_data = [
    (1, "Delhi", "North"),
    (2, "Mumbai", "West"),
    (3, "Pune", "West"),
    (4, "Mumbai", "West")
]

location_df = spark.createDataFrame(
    location_data,
    ["customer_id", "city", "region"]
)

location_df.show()

In [ ]:
multiple_key_join_df = customers_df.join(
    location_df,
    on=["customer_id", "city"],
    how="inner"
)

multiple_key_join_df.show()

When column names differ, write both conditions explicitly and combine them using `&`.

In [ ]:
explicit_multiple_join_df = customers_df.alias("c").join(
    location_df.alias("l"),
    (col("c.customer_id") == col("l.customer_id")) &
    (col("c.city") == col("l.city")),
    "inner"
).select(
    col("c.customer_id"),
    col("c.customer_name"),
    col("c.city"),
    col("l.region")
)

explicit_multiple_join_df.show()

In [ ]:
spark.stop()

In [ ]:
left_null_df = spark.createDataFrame(
    [(1, "A"), (2, "B"), (None, "Missing Left")],
    ["id", "left_value"]
)

right_null_df = spark.createDataFrame(
    [(1, "X"), (3, "Y"), (None, "Missing Right")],
    ["id", "right_value"]
)

left_null_df.show()
right_null_df.show()

In [ ]:
left_null_df.join(
    right_null_df,
    on="id",
    how="inner"
).show()

Use `eqNullSafe()` when null should match null.

In [ ]:
null_safe_join_df = left_null_df.alias("l").join(
    right_null_df.alias("r"),
    col("l.id").eqNullSafe(col("r.id")),
    "inner"
).select(
    col("l.id"),
    col("l.left_value"),
    col("r.right_value")
)

null_safe_join_df.show()

In [ ]:
print("Customer rows:", customers_df.count())
print("Order rows:", orders_df.count())
print("Inner join rows:", inner_join_df.count())
print("Left join rows:", left_join_df.count())

In [ ]:
spark.stop()